1.comparaison entre images et leur corespondance (labels)

In [12]:
#comparaison entre les images et les labels pour identifier les images sans annotation et les annotations sans image correspondante
from pathlib import Path

DATASET_DIR = Path("../trafic_data")

for split in ["train", "valid"]:
    images = list((DATASET_DIR / split / "images").glob("*"))
    labels = list((DATASET_DIR / split / "labels").glob("*.txt"))

    noms_images = {f.stem for f in images}   #f.stem récupère le nom du fichier sans extension.
    noms_labels = {f.stem for f in labels}

    print(f"\n{split.upper()} :")
    print("Images sans Labels :", len(noms_images - noms_labels))   #trouve les images sans Label correspondante
    print("label sans image :", len(noms_labels - noms_images))   #trouve labels sans image correspondante


TRAIN :
Images sans Labels : 0
label sans image : 0

VALID :
Images sans Labels : 0
label sans image : 0


2.Examiner le format YOLO

In [4]:
#afficher le nombre d'annotations vides pour chaque split
for split in ["train", "valid"]:
    labels = list((DATASET_DIR / split / "labels").glob("*.txt"))
    vides = [f for f in labels if f.stat().st_size == 0]

    print(f"{split.upper()} : {len(vides)} annotations vides")
    
    
#found2 in Train 

TRAIN : 0 annotations vides
VALID : 0 annotations vides


In [5]:
#Afficher la structure du fichier text d'annotation
label_path = next(
    fichier for fichier in (DATASET_DIR / "train" / "labels").glob("*.txt")
    if fichier.stat().st_size > 0    
)

print("Fichier :", label_path.name)
print(label_path.read_text())

Fichier : 02_jpg.rf.65a084066fc353cd023eb5c953f40efe.txt
17 0.9546875 0.6892523364485982 0.0859375 0.11682242990654206
19 0.721875 0.5607476635514018 0.0953125 0.11448598130841121
17 0.3671875 0.6635514018691588 0.05625 0.10514018691588785
10 0.2828125 0.48130841121495327 0.0171875 0.037383177570093455
10 0.4015625 0.5327102803738317 0.01875 0.04906542056074766
10 0.675 0.4766355140186916 0.0234375 0.0397196261682243
10 0.696875 0.4602803738317757 0.0171875 0.030373831775700934
10 0.3390625 0.3317757009345794 0.0078125 0.014018691588785047
10 0.4015625 0.308411214953271 0.00625 0.009345794392523364
10 0.478125 0.3317757009345794 0.0078125 0.016355140186915886
16 0.2984375 0.3107476635514019 0.0140625 0.016355140186915886
11 0.315625 0.308411214953271 0.0140625 0.02336448598130841
4 0.2828125 0.29906542056074764 0.01875 0.037383177570093455
4 0.2796875 0.2733644859813084 0.0171875 0.014018691588785047
17 0.3296875 0.4369158878504673 0.021875 0.04672897196261682
17 0.30625 0.380841121495

In [6]:
valeurs = label_path.read_text().splitlines()[0].split()

print("Value numbers :", len(valeurs))
print("Class ID :", valeurs[0])
print("X center  :", valeurs[1])
print("Y center :", valeurs[2])
print("width :", valeurs[3])
print("height :", valeurs[4])

Value numbers : 5
Class ID : 17
X center  : 0.9546875
Y center : 0.6892523364485982
width : 0.0859375
height : 0.11682242990654206


In [7]:
#vérifie la normalisation des coordonnées et des dimensions des bounding boxes au format YOLO.
x, y, largeur, hauteur = map(float, valeurs[1:]) # Convertir les valeurs en nombres

print("Centre X valide :", 0 <= x <= 1)
print("Centre Y valide :", 0 <= y <= 1)
print("Largeur valide :", 0 < largeur <= 1)
print("Hauteur valide :", 0 < hauteur <= 1)

Centre X valide : True
Centre Y valide : True
Largeur valide : True
Hauteur valide : True


In [13]:
#Vérifier le fichier data.yaml, les noms des classes et les identifiants utilisés dans les annotations
for split in ["train", "valid"]:
    class_ids = set()

    for label_file in (DATASET_DIR / split / "labels").glob("*.txt"):
        for ligne in label_file.read_text().splitlines():
            if ligne.strip():
                class_ids.add(int(ligne.split()[0]))


    print(f"\n{split.upper()}")
    print("Identifiants utilisés :", sorted(class_ids))


TRAIN
Identifiants utilisés : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

VALID
Identifiants utilisés : [3, 4, 5, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [14]:
# Vérifier les images corrompues
from PIL import Image

for split in ["train", "valid"]:
    dossier = DATASET_DIR / split / "images"
    corrompues = []

    for fichier in dossier.iterdir():
        if fichier.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
            with Image.open(fichier) as img:
                img.verify()

    print(split.upper(), ":", len(corrompues), "images corrompues")

TRAIN : 0 images corrompues
VALID : 0 images corrompues


In [10]:
# Identifier les lignes invalides
for split in ["train", "valid"]:
    labels_dir = DATASET_DIR / split / "labels"
    invalides = []

    for fichier in labels_dir.glob("*.txt"):
        for n, ligne in enumerate(fichier.read_text().splitlines(), 1):
            if not ligne.strip():
                continue

            v = ligne.split()

            if len(v) != 5:
                invalides.append(f"{fichier.name}, ligne {n}")
                continue

            c, x, y, w, h = map(float, v)

            if not (c.is_integer() and 0 <= c < 21 and
                    0 <= x <= 1 and 0 <= y <= 1 and
                    0 < w <= 1 and 0 < h <= 1):
                invalides.append(f"{fichier.name}, ligne {n}")

    print(split, ":", len(invalides), "lignes invalides")
    print(invalides[:10])

train : 0 lignes invalides
[]
valid : 24 lignes invalides
['Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 1', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 2', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 3', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 4', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 5', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 6', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 7', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 8', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 9', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 10']


In [11]:
# Supprimer les labels vides et leurs images
for split in ["train", "valid"]:
    labels_dir = DATASET_DIR / split / "labels"
    images_dir = DATASET_DIR / split / "images"
    supprimes = 0

    for label in labels_dir.glob("*.txt"):
        if not label.read_text(encoding="utf-8").strip():
            nom_image = label.stem

            for ext in [".jpg", ".jpeg", ".png", ".webp"]:
                image = images_dir / (nom_image + ext)

                if image.exists():
                    image.unlink()
                    break

            label.unlink()
            supprimes += 1

    print(split.upper(), ":", supprimes, "labels vides Deleted ")

TRAIN : 0 labels vides Deleted 
VALID : 0 labels vides Deleted 
